Some of the images in the lofar dataset are empty but we have the labels for those images as well so while processing the data, make sure remove the images which are empty.

Also the lofar images are not present in the catalouge for night time, but still the new dataset has data for that particular times. So the noaa is also labelling that data timestamps. Ask pietro if i should exclude these data as they might not have relevant information. Ask why do we have the nighttime data for lofar??

 So look at the average start and end times of the sun images in the catalouge an remove the data images for the data for those nighttime images

The labelling with noaa is pretty decent for type 3 and 5. For type 2 it misses some images as the time range of the burst is a bit long. For type 4 its the worst as the time range in noaa is huge.

The problem with labelling that i have done in solar_events_cleaned analysis is that it wasnt able to label all the instances properly as sometimes the bursts recorded in noaa are longer than 15 minutes, which is the length of the lofar image. We can tackle this by using the start and end time in noaa burst and label all the image which are in this range. Which means that, for a parttcular date, we will start looking from the first image (we will not approximate the time in the timestamps) of lofar, and if an a burst from noaa comes in the 15 minute time ie between the time of two consecutive timestamps, we will label the image of that timestamp as a burst. But if the length of burst is longer than 15 minutes, then we will label all the timestamps (2 or even more), with the single noaa burst. 

This will help in preserving the original name of the timestamp as well as solve the problem of iregular labelling. 

Some type 4 are more than 7 h hours long in noaa but on inspecting lofar data, they are no seen the full period. Maybe to solve this we can use a time range only to label those images (for eg, for time reange of noaa burst more than 75 minutes, keep give labels only to the images to 5 images ie image timstamps of lofar in 75 minutes time), but still there's a possibility of not having the burst in the multiple labeled images. Like some will have a burst in those images but theres a chance so will not. Maybe we can solve this using MIL machine learning technique. 

For more images we can use GAN for the best representation of the burst images. Like we will plot all those images and see if we can manually select the best images for increasing dataset using GAN. 

### Lets label the noaa data from solar_events_cleaned file. 

In [8]:
import json
import pandas as pd
import re

# --- Fetch "date", "begin", "end", "particulars" from solar_events_cleaned.json ---
json_path = "USED Fetch data from NOAA (works)/solar_events_cleaned.json"

with open(json_path, "r") as f:
    events = json.load(f)

# If the JSON is a dict with a list inside, extract the list
if isinstance(events, dict):
    for v in events.values():
        if isinstance(v, list):
            events = v
            break

event_rows = []
for row in events:
    try:
        event_rows.append({
            "date": row["date"],
            "begin": row["begin"],
            "end": row["end"],
            "particulars": row["particulars"]
        })
    except KeyError:
        continue

events_df = pd.DataFrame(event_rows)
print("Events DataFrame:")
display(events_df.head())

# --- Fetch timestamps and keys from dates_and_urls_of_new_data.csv ---
csv_path = "dates_and_urls_of_new_data.csv"
timestamps_df = pd.read_csv(csv_path)

# Extract date, time, and keep the original key
def extract_date_time_key(s):
    # Example: 2044342_2024-08-14_11:20:00.000000
    match = re.search(r"(\d{4}-\d{2}-\d{2})[_T ](\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?)", str(s))
    if match:
        return match.group(1), match.group(2), s
    return None, None, s

# Extract date, time, and keep the original key
def extract_date_time_key(s):
    # Example: 2044342_2024-08-14_11:20:00.000000
    match = re.search(r"(\d{4}-\d{2}-\d{2})[_T ](\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?)", str(s))
    if match:
        return match.group(1), match.group(2), s
    return None, None, s

# Try to find the column with the timestamp key
timestamp_col = None
for col in timestamps_df.columns:
    if timestamps_df[col].astype(str).str.contains(r"\d{4}-\d{2}-\d{2}").any():
        timestamp_col = col
        break

if timestamp_col is None:
    raise ValueError("No column with timestamp found in CSV.")

timestamps_df[["date", "time", "key"]] = timestamps_df[timestamp_col].apply(lambda s: pd.Series(extract_date_time_key(s)))

# Sort by date and time
timestamps_df["date_dt"] = pd.to_datetime(timestamps_df["date"])
timestamps_df["time_dt"] = pd.to_timedelta(timestamps_df["time"])
timestamps_df = timestamps_df.sort_values(["date_dt", "time_dt"]).reset_index(drop=True)
timestamps_df = timestamps_df.drop(columns=["date_dt", "time_dt"])

print("Timestamps DataFrame (date, time, and key extracted, sorted):")
display(timestamps_df[["date", "time", "key"]].head())

Events DataFrame:


,date,begin,end,particulars
0,2022-05-02,0000,0240,III/1
1,2022-05-02,0249,0250,III/2
2,2022-05-02,0306,0306,III/1
3,2022-05-02,0342,0343,III/1
4,2022-05-02,0415,0415,III/2


Timestamps DataFrame (date, time, and key extracted, sorted):


,date,time,key
0,2022-05-02,06:30:00.000000,858918_2022-05-02_06:30:00.000000
1,2022-05-02,06:45:00.000000,858918_2022-05-02_06:45:00.000000
2,2022-05-02,07:00:00.000000,858918_2022-05-02_07:00:00.000000
3,2022-05-02,07:15:00.000000,858918_2022-05-02_07:15:00.000000
4,2022-05-02,07:30:00.000000,858918_2022-05-02_07:30:00.000000


In [5]:
import numpy as np
from datetime import datetime, timedelta

# Helper: convert string to datetime (date + time)
def to_datetime(date_str, time_str):
    try:
        return datetime.strptime(f"{date_str} {time_str}", "%Y-%m-%d %H:%M:%S.%f")
    except ValueError:
        return datetime.strptime(f"{date_str} {time_str}", "%Y-%m-%d %H:%M:%S")

# Prepare a column for labels
df["label"] = ""

# Group timestamps by date for fast lookup
date_to_times = {}
date_to_indices = {}
for date, group in df.groupby("date"):
    times = group["time"].tolist()
    indices = group.index.tolist()
    dtimes = [to_datetime(date, t) for t in times]
    date_to_times[date] = dtimes
    date_to_indices[date] = indices

for _, event in events_df.iterrows():
    date = event["date"]
    if date not in date_to_times:
        continue

    begin_dt = to_datetime(date, event["begin"])
    end_dt = to_datetime(date, event["end"])
    particulars = event["particulars"]
    times = date_to_times[date]
    indices = date_to_indices[date]

    # Find all timestamps where begin_dt falls within [timestamp, timestamp+15min)
    label_indices = []
    for i, t in enumerate(times):
        if t <= begin_dt < t + timedelta(minutes=15):
            label_indices.append(i)

    # If burst is longer than 1 hour, also label the next 3 timestamps (total 4), but only if still same date
    duration = (end_dt - begin_dt).total_seconds() / 60  # in minutes
    if duration > 60 and label_indices:
        anchor = label_indices[0]
        for j in range(1, 4):
            idx = anchor + j
            if idx < len(times):
                # Only label if still the same date
                if times[idx].date().isoformat() == date:
                    label_indices.append(idx)
                else:
                    break

    # Assign label (append if already present)
    for idx in set(label_indices):
        row = indices[idx]
        if df.at[row, "label"]:
            labels_set = set(df.at[row, "label"].split(";"))
            labels_set.add(particulars)
            df.at[row, "label"] = ";".join(sorted(labels_set))
        else:
            df.at[row, "label"] = particulars

print("Labeling complete. Example labeled timestamps:")
display(df[df["label"] != ""].head(10))

ValueError: time data '2022-05-02 0000' does not match format '%Y-%m-%d %H:%M:%S'

In [3]:
import pandas as pd

def process_events(events_df, timestamps_df):
    # Create datetime columns for timestamps
    timestamps_df['timestamp_dt'] = pd.to_datetime(timestamps_df['date'] + ' ' + timestamps_df['time'])
    
    # Preprocess events: convert begin and end to datetime
    events_df = events_df.copy()
    events_df['begin_time'] = events_df['begin'].apply(lambda x: f"{x[:2]}:{x[2:]}")
    events_df['end_time'] = events_df['end'].apply(lambda x: f"{x[:2]}:{x[2:]}")
    events_df['begin_dt'] = pd.to_datetime(events_df['date'] + ' ' + events_df['begin_time'], errors='coerce')
    events_df['end_dt'] = pd.to_datetime(events_df['date'] + ' ' + events_df['end_time'], errors='coerce')
    
    # Initialize label column in timestamps
    timestamps_df = timestamps_df.copy()
    timestamps_df['label'] = None
    
    # Group timestamps by date for faster lookup
    timestamps_by_date = {date: group for date, group in timestamps_df.groupby('date')}
    
    for idx, event in events_df.iterrows():
        if pd.isnull(event['begin_dt']) or pd.isnull(event['end_dt']):
            continue
            
        event_date = event['date']
        event_begin_dt = event['begin_dt']
        event_end_dt = event['end_dt']
        duration_minutes = (event_end_dt - event_begin_dt).total_seconds() / 60.0
        
        if duration_minutes < 0:
            continue
        
        if event_date not in timestamps_by_date:
            continue
        
        date_timestamps = timestamps_by_date[event_date]
        cond_start = (event_begin_dt >= date_timestamps['timestamp_dt']) & \
                     (event_begin_dt < date_timestamps['timestamp_dt'] + pd.Timedelta(minutes=15))
        matching = date_timestamps[cond_start]
        
        if matching.empty:
            continue
            
        start_idx = matching.index[0]
        date_indices = date_timestamps.index.tolist()
        pos = date_indices.index(start_idx)
        
        if duration_minutes > 60:
            indices_to_label = date_indices[pos:pos+4] if pos + 4 <= len(date_indices) else date_indices[pos:]
        else:
            indices_to_label = []
            current_pos = pos
            while current_pos < len(date_indices) and date_timestamps.iloc[current_pos]['timestamp_dt'] <= event_end_dt:
                indices_to_label.append(date_indices[current_pos])
                current_pos += 1
        
        timestamps_df.loc[indices_to_label, 'label'] = event['particulars']
    
    return timestamps_df

In [4]:
# Assuming you have the function defined (from previous solution)
result_df = process_events(events_df, timestamps_df)

In [5]:
# Save full results
result_df.to_csv('labeled_timestamps.csv', index=False)

# Save only labeled entries
result_df[result_df['label'].notnull()].to_csv('labeled_events.csv', index=False)

In [6]:
import pandas as pd
import re

# Load the labeled events CSV
df = pd.read_csv('labeled_events.csv')

# Function to convert Roman numerals to integers
def roman_to_int(roman):
    roman_numeral_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5}
    # Extract the roman numeral part before any "/"
    match = re.match(r'([IVX]+)', roman)
    if match:
        return roman_numeral_map.get(match.group(1), None)
    return None

# Function to process the label column (handles multiple labels separated by ";")
def process_label(label):
    if pd.isnull(label):
        return None
    # Remove anything after "/" and convert Roman to int
    labels = [roman_to_int(part.split('/')[0]) for part in label.split(';')]
    # Return as string for sorting/grouping
    return ';'.join(str(l) for l in labels if l is not None)

# Apply the conversion
df['label_int'] = df['label'].apply(process_label)

# Sort by label_int
df_sorted = df.sort_values('label_int').reset_index(drop=True)

# Save the sorted DataFrame
df_sorted.to_csv('labeled_events_sorted.csv', index=False)

# Preview
print(df_sorted[['date', 'time', 'key', 'label', 'label_int']].head())

         date             time                                 key  \
0  2024-08-08  20:11:00.000000  2043847_2024-08-08_20:11:00.000000   
1  2024-03-23  01:08:00.000000  2029104_2024-03-23_01:08:00.000000   
2  2024-07-22  09:28:00.000000  2042328_2024-07-22_09:28:00.000000   
3  2023-07-16  19:12:00.000000  2022430_2023-07-16_19:12:00.000000   
4  2023-07-07  09:58:00.000000  2021311_2023-07-07_09:58:00.000000   

         label label_int  
0  II/2    359         2  
1  II/2    791         2  
2  II/2    536         2  
3  II/2    567         2  
4  II/1    601         2  


We will make some changes to the logic of the code. So we know that the timestamps_df is sequentially arranged by date and time. The interval between each instance of the timestamps_df in 15 minutes. So that means if we want to use the events_df which has labels (in particulars column) to label the instances of timestamps_df, using the dates and times values. 

So let's talk about an example to understand what we want the code to do. We want to label the instances using begin and end time of a solar burst (label) from the events_df to timestamps_df. SO if for date 2022-05-02, if a burst occurred at begin=1202 and end=1205, the timestamp which should be labeled in timestamps_df should be key=858918_2022-05-02_12:00:00.000000 since the burst occurs and ends under the 15 minutes from 12:00:000000. Since each timestamp has an corresponding image of which has the burst signal which is 15 minutes long.

Similarly, another example, for a burst in events_df, for date 2022-05-21, a burst happened at begin=1202 and end=1218, then the timestamps which should be labeled in timestamps_df are 861426_2022-05-21_12:00:00.000000 and 861426_2022-05-21_12:15:00.000000. We labeled them both for the same label of the same instance from events_df because the burst lasted for more than 15 minutes from 1202. If it was less then we only would have labeled 861426_2022-05-21_12:00:00.000000. Since the burst lasted longer than 15 minutes in this case, the corresponding images containing the burst will use up 2 timestamps because they will create 2 images. 

So the code should kind of look into each instances in the timestamp_df and label it one by if the burst occurs that timestamp or not because each timestamp has a 15 minute long image. And if the burst length is longer than 15 minutes then multiple timestamps should be labeled based on the begin and end time from the events_df. 

But also again make sure not to label too many timestamps as some of the burst last more than few hours. This will end up over labelling. So we still want to keep a threshold for the difference between begin and end time. If the difference is more than 75 minutes for a specific instance in events_df, then we would label only the timestamps which comes in those 75 minutes. You can maybe do this by making a variable in the for loop, that for the current instance in events_df, if the difference between end and begin is greater than 75, then only label the current timestamp which is about to be labeled plus the next 5 timestamps instances. Just make sure that there is double entry or duplicate entry in the timestamp_df otherwise it will label the same timestamp which has the same image of 15 minutes multiple times instead of labelling the next 15 minute timestamp. 

Save this list in a csv file with the begin and end times as well with dates from events_df which was used to label the timestamp

In [9]:
import pandas as pd

def label_timestamps(events_df, timestamps_df):
    """
    Match solar burst events to 15-minute timestamp intervals
    with special handling for long-duration events (>75 minutes)
    """
    # Create working copies to avoid modifying original DataFrames
    events_df = events_df.copy()
    timestamps_df = timestamps_df.copy()
    
    # Convert event times to proper datetime format
    events_df['begin_time'] = events_df['begin'].apply(lambda x: f"{x[:2]}:{x[2:]}")
    events_df['end_time'] = events_df['end'].apply(lambda x: f"{x[:2]}:{x[2:]}")
    events_df['begin_dt'] = pd.to_datetime(events_df['date'] + ' ' + events_df['begin_time'], errors='coerce')
    events_df['end_dt'] = pd.to_datetime(events_df['date'] + ' ' + events_df['end_time'], errors='coerce')
    
    # Create timestamp intervals (15-minute windows)
    timestamps_df['start_interval'] = pd.to_datetime(timestamps_df['date'] + ' ' + timestamps_df['time'])
    timestamps_df['end_interval'] = timestamps_df['start_interval'] + pd.Timedelta(minutes=15)
    
    # Prepare results storage
    results = []
    
    # Process each event
    for _, event in events_df.iterrows():
        # Skip events with invalid dates/times
        if pd.isnull(event['begin_dt']) or pd.isnull(event['end_dt']):
            continue
            
        event_date = event['date']
        event_begin = event['begin_dt']
        event_end = event['end_dt']
        duration = (event_end - event_begin).total_seconds() / 60
        
        # Skip negative durations
        if duration < 0:
            continue
            
        # Get timestamps for current date
        date_timestamps = timestamps_df[timestamps_df['date'] == event_date]
        if date_timestamps.empty:
            continue
            
        # Case 1: Long events (>75 minutes)
        if duration > 75:
            # Find starting timestamp interval
            start_mask = (
                (event_begin >= date_timestamps['start_interval']) & 
                (event_begin < date_timestamps['end_interval'])
            )
            start_intervals = date_timestamps[start_mask]
            
            if not start_intervals.empty:
                start_idx = start_intervals.index[0]
                date_indices = date_timestamps.index.tolist()
                
                # Get position of starting index
                try:
                    pos = date_indices.index(start_idx)
                    # Get next 5 intervals (total of 6 intervals)
                    end_pos = min(pos + 6, len(date_indices))
                    for i in range(pos, end_pos):
                        idx = date_indices[i]
                        results.append({
                            'timestamp_key': timestamps_df.at[idx, 'key'],
                            'timestamp_date': timestamps_df.at[idx, 'date'],
                            'timestamp_time': timestamps_df.at[idx, 'time'],
                            'event_date': event['date'],
                            'event_begin': event['begin'],
                            'event_end': event['end'],
                            'event_particulars': event['particulars']
                        })
                except ValueError:
                    pass
        
        # Case 2: Short/medium events (≤75 minutes)
        else:
            # Find all overlapping intervals
            overlap_mask = (
                (event_begin < date_timestamps['end_interval']) & 
                (event_end > date_timestamps['start_interval'])
            )
            overlapping = date_timestamps[overlap_mask]
            
            for idx, row in overlapping.iterrows():
                results.append({
                    'timestamp_key': row['key'],
                    'timestamp_date': row['date'],
                    'timestamp_time': row['time'],
                    'event_date': event['date'],
                    'event_begin': event['begin'],
                    'event_end': event['end'],
                    'event_particulars': event['particulars']
                })
    
    return pd.DataFrame(results)

# Load your data (replace with actual paths if reading from files)
events_df = events_df.copy()

timestamps_df = timestamps_df.copy()


# Process and save results
results_df = label_timestamps(events_df, timestamps_df)
results_df.to_csv('burst_event_matches.csv', index=False)

print(f"Saved {len(results_df)} matches to burst_event_matches.csv")
print("Sample results:")
print(results_df.head())

Saved 4325 matches to burst_event_matches.csv
Sample results:
                       timestamp_key timestamp_date   timestamp_time  \
0  858918_2022-05-02_06:45:00.000000     2022-05-02  06:45:00.000000   
1  858918_2022-05-02_09:15:00.000000     2022-05-02  09:15:00.000000   
2  858918_2022-05-02_09:30:00.000000     2022-05-02  09:30:00.000000   
3  858918_2022-05-02_10:15:00.000000     2022-05-02  10:15:00.000000   
4  859062_2022-05-03_07:00:00.000000     2022-05-03  07:00:00.000000   

   event_date event_begin event_end event_particulars  
0  2022-05-02        0649      0650             III/2  
1  2022-05-02        0929      0934             III/1  
2  2022-05-02        0929      0934             III/1  
3  2022-05-02        1020      1020             III/1  
4  2022-05-03        0704      0705             III/1  
